# 05 — DistilBERT: Controlled SHAP-Guided Model Refinement

**Research project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives  
**Experiment:** Controlled refinement of the saved DistilBERT baseline

This notebook presents a controlled SHAP-guided refinement experiment. The aim was to reduce false-positive predictions while preserving the detection of phishing emails. It reports the recorded training design and evaluation results without changing the official baseline.

## 1. Research objective

The official DistilBERT model incorrectly classified **31 legitimate emails as phishing** and **22 phishing emails as legitimate**. A previous attribution-regularization experiment reduced false negatives slightly but increased false positives. This experiment therefore examined whether a more conservative refinement of the saved baseline model could improve the balance between the two error types.

The acceptance criterion remained **fewer than 31 false positives, with no more than 22 false negatives**.

## 2. Dataset and classification convention

**Table 5.1. Dataset summary**

| Dataset component | Number of emails |
|:---|---:|
| Cleaned dataset | 53,577 |
| Training set | 42,861 |
| Test set | 10,716 |

The class labels were **0 = legitimate** and **1 = phishing**. The project uses the following classification-count notation throughout the notebooks: **TP** = correctly classified legitimate emails; **FP** = legitimate emails classified as phishing; **FN** = phishing emails classified as legitimate; and **TN** = correctly classified phishing emails.

## 3. Controlled refinement method

The refinement began with the **saved official baseline checkpoint**, rather than a fresh pretrained model. The embeddings and the first four transformer layers were frozen. Training was restricted to transformer layers 4 and 5, the pre-classifier, and the classifier. The experiment used a SHAP-guided attribution penalty alongside the classification loss.

No emails were masked, duplicated, or edited during this experiment. The official baseline checkpoint was retained separately.

**Table 5.2. Controlled refinement settings**

| Training setting | Recorded value |
|:---|:---|
| Starting model | Saved official DistilBERT baseline |
| Frozen components | Embeddings; transformer layers 0–3 |
| Trainable components | Transformer layers 4–5; pre-classifier; classifier |
| Refinement epochs | 1 |
| Learning rate | 5 × 10⁻⁶ (0.000005) |
| Physical batch size | 2 |
| Gradient accumulation steps | 4 |
| Effective batch size | 8 |
| SHAP attribution penalty coefficient | 1.0 |
| Classification labels | 0 = legitimate; 1 = phishing |

The small learning rate and limited trainable layers were intended to restrict the size of changes to the baseline model. The effective batch size was eight because gradients from four physical batches of two emails were accumulated before an optimizer update.

## 4. SHAP-guided feature selection

The refinement used a fixed set of **20 tokens identified during the earlier SHAP analysis**. These tokens guided the attribution penalty; they were not used as new class labels, and the email text was not rewritten in this experiment.

**Table 5.3. Tokens used to guide attribution regularization**

| Group | Token 1 | Token 2 | Token 3 | Token 4 |
|:---:|:---|:---|:---|:---|
| 1 | symbol | occur | hard | cut |
| 2 | his | dear | glass | success |
| 3 | general | company | had | now |
| 4 | investors | bit | tech | got |
| 5 | my | brand | told | sun |

The groups are used only to present the tokens clearly. They do not represent importance ranks or SHAP attribution values.

## 5. Evaluation results

The saved refined model was evaluated on the same 10,716-email test split used for the official baseline. The decision threshold and the test data were not modified during this evaluation.

**Table 5.4. Comparison of classification outcomes**

| Classification outcome | Official baseline | Controlled refinement | Change |
|:---|---:|---:|---:|
| TP — correctly classified legitimate | 4,717 | 4,720 | +3 |
| FP — legitimate classified as phishing | **31** | **28** | **−3** |
| FN — phishing classified as legitimate | **22** | **27** | **+5** |
| TN — correctly classified phishing | 5,946 | 5,941 | −5 |
| **Total emails** | **10,716** | **10,716** | **0** |

The controlled refinement reduced false positives by three cases but increased false negatives by five cases. Consequently, it did **not** meet the study's requirement to reduce false positives without increasing false negatives.

## 6. Performance verification

The code below calculates accuracy, phishing-class precision, recall, and F1-score directly from the recorded classification counts. It does not retrain or re-evaluate the saved model. Under the project's TP/TN naming convention, phishing precision uses `TN / (TN + FP)` and phishing recall uses `TN / (TN + FN)`.

In [ ]:
import pandas as pd

recorded = {
    "Official baseline": {"TP": 4717, "FP": 31, "FN": 22, "TN": 5946},
    "Controlled refinement": {"TP": 4720, "FP": 28, "FN": 27, "TN": 5941},
}

rows = []
for model, c in recorded.items():
    total = sum(c.values())
    assert total == 10716, f"Unexpected test-set size for {model}: {total}"
    precision = c["TN"] / (c["TN"] + c["FP"])
    recall = c["TN"] / (c["TN"] + c["FN"])
    rows.append({
        "Model": model,
        "Accuracy": (c["TP"] + c["TN"]) / total,
        "Precision (phishing)": precision,
        "Recall (phishing)": recall,
        "F1-score (phishing)": 2 * precision * recall / (precision + recall),
        "FP": c["FP"], "FN": c["FN"],
    })

metrics = pd.DataFrame(rows).set_index("Model")
print(metrics.to_string(float_format=lambda x: f"{x:.6f}"))
assert recorded["Controlled refinement"]["FP"] < recorded["Official baseline"]["FP"]
assert recorded["Controlled refinement"]["FN"] > recorded["Official baseline"]["FN"]


**Table 5.5. Classification performance comparison**

| Metric | Official baseline | Controlled refinement |
|:---|---:|---:|
| Accuracy | 0.995054 | 0.994867 |
| Precision (phishing) | 0.994813 | 0.995309 |
| Recall (phishing) | 0.996314 | 0.995476 |
| F1-score (phishing) | 0.995563 | 0.995392 |

All four metrics in Table 5.5 are calculated from the recorded confusion-matrix counts. The original baseline ROC-AUC was **0.999262**. The refined model's ROC-AUC is not reported because its prediction probabilities are not included in this notebook.

## 7. Interpretation and conclusion

Controlled refinement reduced false positives from **31 to 28**, an improvement of three cases. However, false negatives increased from **22 to 27**, meaning that five additional phishing emails were incorrectly classified as legitimate. This result did not satisfy the research requirement of reducing false positives without increasing false negatives.

The refined checkpoint was therefore **not selected as the final model**. The findings informed the next experiments, which examined other ways to improve legitimate-email classification while preserving phishing detection.

**Reproducibility:** The notebook presents the recorded experiment settings and results, with executable checks for the performance metrics. The original model training is not repeated here.